In [7]:
import torch
import torch.nn as nn

class CNNModel(nn.Module):
    def __init__(self, input_channels, D):
        super(CNNModel, self).__init__()
        self.layer1 = nn.BatchNorm1d(input_channels)
        self.layer2 = nn.Conv1d(input_channels, D, kernel_size=3, padding=1)
        self.layer3 = nn.ReLU()
        self.layer4 = nn.BatchNorm1d(D)
        self.layer5 = nn.MaxPool1d(2)
        self.layer6 = nn.Conv1d(D, 2 * D, kernel_size=3, padding=1)
        self.layer7 = nn.ReLU()
        self.layer8 = nn.BatchNorm1d(2 * D)
        self.layer9 = nn.MaxPool1d(2)
        self.layer10 = nn.Conv1d(2 * D, 4 * D, kernel_size=3, padding=1)
        self.layer11 = nn.ReLU()
        self.layer12 = nn.BatchNorm1d(4 * D)
    def forward(self, x):
        print("Input:", x.shape)
        x = self.layer1(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer2(x)
        print("After Conv1d (input -> D):", x.shape)
        x = self.layer3(x)
        print("After ReLU:", x.shape)
        x = self.layer4(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer5(x)
        print("After MaxPool1d(2):", x.shape)
        x = self.layer6(x)
        print("After Conv1d (D -> 2D):", x.shape)
        x = self.layer7(x)
        print("After ReLU:", x.shape)
        x = self.layer8(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer9(x)
        print("After MaxPool1d(2):", x.shape)
        x = self.layer10(x)
        print("After Conv1d (2D -> 4D):", x.shape)
        x = self.layer11(x)
        print("After ReLU:", x.shape)
        x = self.layer12(x)
        print("After BatchNorm1d:", x.shape)
        return x
# Example usage
batch_size = 8
input_channels = 2  # Number of input channels
sequence_length = 18
D = 16
model = CNNModel(input_channels, D)
x = torch.randn(batch_size, input_channels, sequence_length)
output = model(x)

from torchinfo import summary
summary(model, input_size=((8, 2, 18)))


Input: torch.Size([8, 2, 18])
After BatchNorm1d: torch.Size([8, 2, 18])
After Conv1d (input -> D): torch.Size([8, 16, 18])
After ReLU: torch.Size([8, 16, 18])
After BatchNorm1d: torch.Size([8, 16, 18])
After MaxPool1d(2): torch.Size([8, 16, 9])
After Conv1d (D -> 2D): torch.Size([8, 32, 9])
After ReLU: torch.Size([8, 32, 9])
After BatchNorm1d: torch.Size([8, 32, 9])
After MaxPool1d(2): torch.Size([8, 32, 4])
After Conv1d (2D -> 4D): torch.Size([8, 64, 4])
After ReLU: torch.Size([8, 64, 4])
After BatchNorm1d: torch.Size([8, 64, 4])
Input: torch.Size([8, 2, 18])
After BatchNorm1d: torch.Size([8, 2, 18])
After Conv1d (input -> D): torch.Size([8, 16, 18])
After ReLU: torch.Size([8, 16, 18])
After BatchNorm1d: torch.Size([8, 16, 18])
After MaxPool1d(2): torch.Size([8, 16, 9])
After Conv1d (D -> 2D): torch.Size([8, 32, 9])
After ReLU: torch.Size([8, 32, 9])
After BatchNorm1d: torch.Size([8, 32, 9])
After MaxPool1d(2): torch.Size([8, 32, 4])
After Conv1d (2D -> 4D): torch.Size([8, 64, 4])
Aft

Layer (type:depth-idx)                   Output Shape              Param #
CNNModel                                 [8, 64, 4]                --
├─BatchNorm1d: 1-1                       [8, 2, 18]                4
├─Conv1d: 1-2                            [8, 16, 18]               112
├─ReLU: 1-3                              [8, 16, 18]               --
├─BatchNorm1d: 1-4                       [8, 16, 18]               32
├─MaxPool1d: 1-5                         [8, 16, 9]                --
├─Conv1d: 1-6                            [8, 32, 9]                1,568
├─ReLU: 1-7                              [8, 32, 9]                --
├─BatchNorm1d: 1-8                       [8, 32, 9]                64
├─MaxPool1d: 1-9                         [8, 32, 4]                --
├─Conv1d: 1-10                           [8, 64, 4]                6,208
├─ReLU: 1-11                             [8, 64, 4]                --
├─BatchNorm1d: 1-12                      [8, 64, 4]                128
Total pa

In [24]:
class SmallCNNBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_len, D):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.BatchNorm1d(input_len),
            nn.Conv1d(input_len, D, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(D),
            nn.MaxPool1d(2),
            nn.Conv1d(D, 2 * D, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(2 * D),
            nn.MaxPool1d(2),
            nn.Conv1d(2 * D, 4 * D, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(4 * D),
        )
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(4 * D, input_len)

    # ------------------------------------------------------------------------------------------------------------------
    def forward(self, x):
        x = self.cnn(x)
        x = self.global_avg_pool(x).squeeze(-1)  # [B, C]
        x = self.fc(x)
        return x

# Example usage
batch_size = 8
input_channels = 2  # Number of input channels
sequence_length = 18
D = 16
model = SmallCNNBranch(input_channels, D)
x = torch.randn(batch_size, input_channels, sequence_length)
output = model(x)

from torchinfo import summary
summary(model, input_size=((8, 2, 18)))

Layer (type:depth-idx)                   Output Shape              Param #
SmallCNNBranch                           [8, 2]                    --
├─Sequential: 1-1                        [8, 64, 4]                --
│    └─BatchNorm1d: 2-1                  [8, 2, 18]                4
│    └─Conv1d: 2-2                       [8, 16, 18]               112
│    └─ReLU: 2-3                         [8, 16, 18]               --
│    └─BatchNorm1d: 2-4                  [8, 16, 18]               32
│    └─MaxPool1d: 2-5                    [8, 16, 9]                --
│    └─Conv1d: 2-6                       [8, 32, 9]                1,568
│    └─ReLU: 2-7                         [8, 32, 9]                --
│    └─BatchNorm1d: 2-8                  [8, 32, 9]                64
│    └─MaxPool1d: 2-9                    [8, 32, 4]                --
│    └─Conv1d: 2-10                      [8, 64, 4]                6,208
│    └─ReLU: 2-11                        [8, 64, 4]                --
│    └─Ba

In [28]:
class SmallCNNBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_channels, D):
        super().__init__()
        self.layer1 = nn.BatchNorm1d(input_channels)
        self.layer2 = nn.Conv1d(input_channels, D, kernel_size=3, padding=1)
        self.layer3 = nn.ReLU()
        self.layer4 = nn.BatchNorm1d(D)
        self.layer5 = nn.MaxPool1d(2)
        self.layer6 = nn.Conv1d(D, 2 * D, kernel_size=3, padding=1)
        self.layer7 = nn.ReLU()
        self.layer8 = nn.BatchNorm1d(2 * D)
        self.layer9 = nn.MaxPool1d(2)
        self.layer10 = nn.Conv1d(2 * D, 4 * D, kernel_size=3, padding=1)
        self.layer11 = nn.ReLU()
        self.layer12 = nn.BatchNorm1d(4 * D)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(4 * D, input_channels)

    def forward(self, x):
        print("Input:", x.shape)
        x = self.layer1(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer2(x)
        print("After Conv1d (input -> D):", x.shape)
        x = self.layer3(x)
        print("After ReLU:", x.shape)
        x = self.layer4(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer5(x)
        print("After MaxPool1d(2):", x.shape)
        x = self.layer6(x)
        print("After Conv1d (D -> 2D):", x.shape)
        x = self.layer7(x)
        print("After ReLU:", x.shape)
        x = self.layer8(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer9(x)
        print("After MaxPool1d(2):", x.shape)
        x = self.layer10(x)
        print("After Conv1d (2D -> 4D):", x.shape)
        x = self.layer11(x)
        print("After ReLU:", x.shape)
        x = self.layer12(x)
        print("After BatchNorm1d:", x.shape)
        x = self.global_avg_pool(x).squeeze(-1)
        print("After global_avg_pool:", x.shape)
        x = self.fc(x)
        print("After linear:", x.shape)
        return x

# Example usage
batch_size = 8
input_channels = 2  # Number of input channels
sequence_length = 18
D = 16
model = SmallCNNBranch(input_channels, D)
x = torch.randn(batch_size, input_channels, sequence_length)
output = model(x)

from torchinfo import summary
summary(model, input_size=((8, 2, 18)))

Input: torch.Size([8, 2, 18])
After BatchNorm1d: torch.Size([8, 2, 18])
After Conv1d (input -> D): torch.Size([8, 16, 18])
After ReLU: torch.Size([8, 16, 18])
After BatchNorm1d: torch.Size([8, 16, 18])
After MaxPool1d(2): torch.Size([8, 16, 9])
After Conv1d (D -> 2D): torch.Size([8, 32, 9])
After ReLU: torch.Size([8, 32, 9])
After BatchNorm1d: torch.Size([8, 32, 9])
After MaxPool1d(2): torch.Size([8, 32, 4])
After Conv1d (2D -> 4D): torch.Size([8, 64, 4])
After ReLU: torch.Size([8, 64, 4])
After BatchNorm1d: torch.Size([8, 64, 4])
After global_avg_pool: torch.Size([8, 64])
After linear: torch.Size([8, 2])
Input: torch.Size([8, 2, 18])
After BatchNorm1d: torch.Size([8, 2, 18])
After Conv1d (input -> D): torch.Size([8, 16, 18])
After ReLU: torch.Size([8, 16, 18])
After BatchNorm1d: torch.Size([8, 16, 18])
After MaxPool1d(2): torch.Size([8, 16, 9])
After Conv1d (D -> 2D): torch.Size([8, 32, 9])
After ReLU: torch.Size([8, 32, 9])
After BatchNorm1d: torch.Size([8, 32, 9])
After MaxPool1d(2)

Layer (type:depth-idx)                   Output Shape              Param #
SmallCNNBranch                           [8, 2]                    --
├─BatchNorm1d: 1-1                       [8, 2, 18]                4
├─Conv1d: 1-2                            [8, 16, 18]               112
├─ReLU: 1-3                              [8, 16, 18]               --
├─BatchNorm1d: 1-4                       [8, 16, 18]               32
├─MaxPool1d: 1-5                         [8, 16, 9]                --
├─Conv1d: 1-6                            [8, 32, 9]                1,568
├─ReLU: 1-7                              [8, 32, 9]                --
├─BatchNorm1d: 1-8                       [8, 32, 9]                64
├─MaxPool1d: 1-9                         [8, 32, 4]                --
├─Conv1d: 1-10                           [8, 64, 4]                6,208
├─ReLU: 1-11                             [8, 64, 4]                --
├─BatchNorm1d: 1-12                      [8, 64, 4]                128
├─Adapti

In [ ]:
class SmallCNNBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_channels, D):
        super().__init__()
        self.layer1 = nn.BatchNorm2d(input_channels)
        self.layer2 = nn.Conv2d(input_channels, D, kernel_size=3, padding=0)
        self.layer3 = nn.ReLU()
        self.layer4 = nn.BatchNorm2d(D)
        self.layer5 = nn.MaxPool2d(2)
        self.layer6 = nn.Conv2d(D, 2 * D, kernel_size=3, padding=1)
        self.layer7 = nn.ReLU()
        self.layer8 = nn.BatchNorm2d(2 * D)
        self.layer9 = nn.MaxPool2d(2)
        self.layer10 = nn.Conv2d(2 * D, 4 * D, kernel_size=3, padding=1)
        self.layer11 = nn.ReLU()
        self.layer12 = nn.BatchNorm2d(4 * D)
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(4 * D, 2 * D)
        self.fc2 = nn.Linear(2 * D, D)
        self.fc3 = nn.Linear(D, input_channels)


    def forward(self, x):
        print("Input:", x.shape)
        x = self.layer1(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer2(x)
        print("After Conv1d (input -> D):", x.shape)
        x = self.layer3(x)
        print("After ReLU:", x.shape)
        x = self.layer4(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer5(x)
        print("After MaxPool1d(2):", x.shape)
        x = self.layer6(x)
        print("After Conv1d (D -> 2D):", x.shape)
        x = self.layer7(x)
        print("After ReLU:", x.shape)
        x = self.layer8(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer9(x)
        print("After MaxPool1d(2):", x.shape)
        x = self.layer10(x)
        print("After Conv1d (2D -> 4D):", x.shape)
        x = self.layer11(x)
        print("After ReLU:", x.shape)
        x = self.layer12(x)
        print("After BatchNorm1d:", x.shape)
        x = self.global_avg_pool(x).squeeze(-1).squeeze(-1)
        print("After global_avg_pool:", x.shape)
        x = self.fc1(x)
        print("After linear1:", x.shape)
        x = self.fc2(x)
        print("After linear2:", x.shape)
        x = self.fc3(x)
        print("After linear3:", x.shape)
        return x

# Example usage
batch_size = 8
input_channels = 2  # Number of input channels
sequence_length = 18
D = 16
model = SmallCNNBranch(input_channels, D)
x = torch.randn(batch_size, input_channels, sequence_length, 33)
output = model(x)

from torchinfo import summary
summary(model, input_size=((8, 2, 18, 33))) # ie batch 8 of 2 profiles of shape (18,33)

Input: torch.Size([8, 2, 18, 33])
After BatchNorm1d: torch.Size([8, 2, 18, 33])
After Conv1d (input -> D): torch.Size([8, 16, 16, 31])
After ReLU: torch.Size([8, 16, 16, 31])
After BatchNorm1d: torch.Size([8, 16, 16, 31])
After MaxPool1d(2): torch.Size([8, 16, 8, 15])
After Conv1d (D -> 2D): torch.Size([8, 32, 8, 15])
After ReLU: torch.Size([8, 32, 8, 15])
After BatchNorm1d: torch.Size([8, 32, 8, 15])
After MaxPool1d(2): torch.Size([8, 32, 4, 7])
After Conv1d (2D -> 4D): torch.Size([8, 64, 4, 7])
After ReLU: torch.Size([8, 64, 4, 7])
After BatchNorm1d: torch.Size([8, 64, 4, 7])
After global_avg_pool: torch.Size([8, 64])
After linear1: torch.Size([8, 32])
After linear2: torch.Size([8, 16])
After linear3: torch.Size([8, 2])
Input: torch.Size([8, 2, 18, 33])
After BatchNorm1d: torch.Size([8, 2, 18, 33])
After Conv1d (input -> D): torch.Size([8, 16, 16, 31])
After ReLU: torch.Size([8, 16, 16, 31])
After BatchNorm1d: torch.Size([8, 16, 16, 31])
After MaxPool1d(2): torch.Size([8, 16, 8, 15])

Layer (type:depth-idx)                   Output Shape              Param #
SmallCNNBranch                           [8, 2]                    --
├─BatchNorm2d: 1-1                       [8, 2, 18, 33]            4
├─Conv2d: 1-2                            [8, 16, 16, 31]           304
├─ReLU: 1-3                              [8, 16, 16, 31]           --
├─BatchNorm2d: 1-4                       [8, 16, 16, 31]           32
├─MaxPool2d: 1-5                         [8, 16, 8, 15]            --
├─Conv2d: 1-6                            [8, 32, 8, 15]            4,640
├─ReLU: 1-7                              [8, 32, 8, 15]            --
├─BatchNorm2d: 1-8                       [8, 32, 8, 15]            64
├─MaxPool2d: 1-9                         [8, 32, 4, 7]             --
├─Conv2d: 1-10                           [8, 64, 4, 7]             18,496
├─ReLU: 1-11                             [8, 64, 4, 7]             --
├─BatchNorm2d: 1-12                      [8, 64, 4, 7]             128
├─Adapt

In [17]:
import torch
import torch.nn as nn


class SmallCNNBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_channels, D):
        super().__init__()
        self.layer1 = nn.BatchNorm3d(input_channels)
        self.layer2 = nn.Conv3d(input_channels, D, kernel_size=3, padding=0)
        self.layer3 = nn.ReLU()
        self.layer4 = nn.BatchNorm3d(D)
        self.layer5 = nn.MaxPool3d(2)
        self.layer6 = nn.Conv3d(D, 2 * D, kernel_size=3, padding=1)
        self.layer7 = nn.ReLU()
        self.layer8 = nn.BatchNorm3d(2 * D)
        self.layer9 = nn.MaxPool3d(2)
        self.layer10 = nn.Conv3d(2 * D, 4 * D, kernel_size=3, padding=1)
        self.layer11 = nn.ReLU()
        self.layer12 = nn.BatchNorm3d(4 * D)
        self.global_avg_pool = nn.AdaptiveAvgPool3d(D)
        self.fc1 = nn.Linear( (4 * D) * (D * D * D) , 2 * D)
        self.fc2 = nn.Linear(2 * D, D)
        self.fc3 = nn.Linear(D, input_channels)


    def forward(self, x):
        print("Input:", x.shape)
        x = self.layer1(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer2(x)
        print("After Conv1d (input -> D):", x.shape)
        x = self.layer3(x)
        print("After ReLU:", x.shape)
        x = self.layer4(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer5(x)
        print("After MaxPool1d(2):", x.shape)
        x = self.layer6(x)
        print("After Conv1d (D -> 2D):", x.shape)
        x = self.layer7(x)
        print("After ReLU:", x.shape)
        x = self.layer8(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer9(x)
        print("After MaxPool1d(2):", x.shape)
        x = self.layer10(x)
        print("After Conv1d (2D -> 4D):", x.shape)
        x = self.layer11(x)
        print("After ReLU:", x.shape)
        x = self.layer12(x)
        print("After BatchNorm1d we want 1 on last :", x.shape)
        x = self.global_avg_pool(x)
        print("After global_avg_pool:", x.shape)
        x = torch.flatten(x, 1)
        print("After flatten:", x.shape)
        x = self.fc1(x)
        print("After linear1:", x.shape)
        x = self.fc2(x)
        print("After linear2:", x.shape)
        x = self.fc3(x)
        print("After linear3:", x.shape)
        return x

# Example usage
batch_size = 8
input_channels = 2  # Number of input channels
sequence_length = 18
T=25

D = 16
model = SmallCNNBranch(input_channels, D)
x = torch.randn(batch_size, input_channels, sequence_length, 33, T)
output = model(x)


from torchinfo import summary
summary(model, input_size=((8, 2, 18, 33, T))) # ie batch 8 of 2 profiles of shape (18,33)

Input: torch.Size([8, 2, 18, 33, 25])
After BatchNorm1d: torch.Size([8, 2, 18, 33, 25])
After Conv1d (input -> D): torch.Size([8, 16, 16, 31, 23])
After ReLU: torch.Size([8, 16, 16, 31, 23])
After BatchNorm1d: torch.Size([8, 16, 16, 31, 23])
After MaxPool1d(2): torch.Size([8, 16, 8, 15, 11])
After Conv1d (D -> 2D): torch.Size([8, 32, 8, 15, 11])
After ReLU: torch.Size([8, 32, 8, 15, 11])
After BatchNorm1d: torch.Size([8, 32, 8, 15, 11])
After MaxPool1d(2): torch.Size([8, 32, 4, 7, 5])
After Conv1d (2D -> 4D): torch.Size([8, 64, 4, 7, 5])
After ReLU: torch.Size([8, 64, 4, 7, 5])
After BatchNorm1d we want 1 on last : torch.Size([8, 64, 4, 7, 5])
After global_avg_pool: torch.Size([8, 64, 16, 16, 16])
After flatten: torch.Size([8, 262144])
After linear1: torch.Size([8, 32])
After linear2: torch.Size([8, 16])
After linear3: torch.Size([8, 2])
Input: torch.Size([8, 2, 18, 33, 25])
After BatchNorm1d: torch.Size([8, 2, 18, 33, 25])
After Conv1d (input -> D): torch.Size([8, 16, 16, 31, 23])
Aft

Layer (type:depth-idx)                   Output Shape              Param #
SmallCNNBranch                           [8, 2]                    --
├─BatchNorm3d: 1-1                       [8, 2, 18, 33, 25]        4
├─Conv3d: 1-2                            [8, 16, 16, 31, 23]       880
├─ReLU: 1-3                              [8, 16, 16, 31, 23]       --
├─BatchNorm3d: 1-4                       [8, 16, 16, 31, 23]       32
├─MaxPool3d: 1-5                         [8, 16, 8, 15, 11]        --
├─Conv3d: 1-6                            [8, 32, 8, 15, 11]        13,856
├─ReLU: 1-7                              [8, 32, 8, 15, 11]        --
├─BatchNorm3d: 1-8                       [8, 32, 8, 15, 11]        64
├─MaxPool3d: 1-9                         [8, 32, 4, 7, 5]          --
├─Conv3d: 1-10                           [8, 64, 4, 7, 5]          55,360
├─ReLU: 1-11                             [8, 64, 4, 7, 5]          --
├─BatchNorm3d: 1-12                      [8, 64, 4, 7, 5]          128
├─Adap

In [12]:
4 * D * D * D * D

262144

In [18]:
import torch

# Define a tensor with a singleton dimension
x = torch.tensor([[1], [2], [3]])

# Expand the tensor to size (3, 4)
expanded_x = x.expand(3, 4)

print(expanded_x)


tensor([[1, 1, 1, 1],
        [2, 2, 2, 2],
        [3, 3, 3, 3]])


In [20]:
import torch
import torch.nn as nn


class SmallCNNBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_channels, D):
        super().__init__()
        self.layer1 = nn.BatchNorm3d(input_channels)
        self.layer2 = nn.Conv3d(input_channels, D, kernel_size=3, padding=0)
        self.layer3 = nn.ReLU()
        self.layer4 = nn.BatchNorm3d(D)
        self.layer5 = nn.MaxPool3d(2)
        self.layer6 = nn.Conv3d(D, 2 * D, kernel_size=3, padding=1)
        self.layer7 = nn.ReLU()
        self.layer8 = nn.BatchNorm3d(2 * D)
        self.layer9 = nn.MaxPool3d(2)
        self.layer10 = nn.Conv3d(2 * D, 4 * D, kernel_size=3, padding=1)
        self.layer11 = nn.ReLU()
        self.layer12 = nn.BatchNorm3d(4 * D)
        self.layer121 = nn.Linear(4 * D, D)
        self.global_avg_pool = nn.AdaptiveAvgPool3d(D)
        self.fc1 = nn.Linear( (4 * D) * (D * D * D) , 2 * D)
        self.fc2 = nn.Linear(2 * D, D)
        self.fc3 = nn.Linear(D, input_channels)


    def forward(self, x):
        print("Input:", x.shape)
        x = self.layer1(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer2(x)
        print("After Conv1d (input -> D):", x.shape)
        x = self.layer3(x)
        print("After ReLU:", x.shape)
        x = self.layer4(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer5(x)
        print("After MaxPool1d(2):", x.shape)
        x = self.layer6(x)
        print("After Conv1d (D -> 2D):", x.shape)
        x = self.layer7(x)
        print("After ReLU:", x.shape)
        x = self.layer8(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer9(x)
        print("After MaxPool1d(2):", x.shape)
        x = self.layer10(x)
        print("After Conv1d (2D -> 4D):", x.shape)
        x = self.layer11(x)
        print("After ReLU:", x.shape)
        x = self.layer12(x)
        print("After BatchNorm1d layer12 we want 1 on last :", x.shape)
        x = self.layer121(x)
        print("After linear 121 :", x.shape)
        x = self.global_avg_pool(x)
        print("After global_avg_pool:", x.shape)
        x = torch.flatten(x, 1)
        print("After flatten:", x.shape)
        x = self.fc1(x)
        print("After linear1:", x.shape)
        x = self.fc2(x)
        print("After linear2:", x.shape)
        x = self.fc3(x)
        print("After linear3:", x.shape)
        return x

# Example usage
batch_size = 8
input_channels = 2  # Number of input channels
sequence_length = 18
T=25

D = 16
model = SmallCNNBranch(input_channels, D)
x = torch.randn(batch_size, input_channels, sequence_length, 33, T)
output = model(x)


from torchinfo import summary
summary(model, input_size=((8, 2, 18, 33, T))) # ie batch 8 of 2 profiles of shape (18,33)

Input: torch.Size([8, 2, 18, 33, 25])
After BatchNorm1d: torch.Size([8, 2, 18, 33, 25])
After Conv1d (input -> D): torch.Size([8, 16, 16, 31, 23])
After ReLU: torch.Size([8, 16, 16, 31, 23])
After BatchNorm1d: torch.Size([8, 16, 16, 31, 23])
After MaxPool1d(2): torch.Size([8, 16, 8, 15, 11])
After Conv1d (D -> 2D): torch.Size([8, 32, 8, 15, 11])
After ReLU: torch.Size([8, 32, 8, 15, 11])
After BatchNorm1d: torch.Size([8, 32, 8, 15, 11])
After MaxPool1d(2): torch.Size([8, 32, 4, 7, 5])
After Conv1d (2D -> 4D): torch.Size([8, 64, 4, 7, 5])
After ReLU: torch.Size([8, 64, 4, 7, 5])
After BatchNorm1d layer12 we want 1 on last : torch.Size([8, 64, 4, 7, 5])


RuntimeError: mat1 and mat2 shapes cannot be multiplied (14336x5 and 64x16)

In [22]:
# 2d input
linear_layer_2d = nn.Linear(in_features=64, out_features=32)
# 1st dimension (128) = batch dimension, input 64 x 64
input_2d = torch.randn(128, 64, 64)
output_2d = linear_layer_2d(input_2d)
print(output_2d.size())
# torch.Size([128, 64, 32])

# 1d input (2d flattened)
linear_layer_1d = nn.Linear(in_features=4096, out_features=32)
# input_1d size = [128, 4096]
input_1d = torch.flatten(input_2d, start_dim=1)
output_1d = linear_layer_1d(input_1d)
print(output_1d.size())
# torch.Size([128, 32])

torch.Size([128, 64, 32])
torch.Size([128, 32])


In [25]:
import torch
import torch.nn as nn

# Input: [B, C, D, H, W]
input_3d = torch.randn(128, 64, 8, 8, 8)

# Linear layer applied per voxel: channels 64 → 32
linear_layer = nn.Linear(in_features=64, out_features=32)

# Rearrange so channel dimension is last
print(input_3d.shape)
x = input_3d.permute(0, 2, 3, 4, 1)  # → [B, D, H, W, C]
print(x.shape)
x = linear_layer(x) # Linear is applied to last dim
print(x.shape)
x = x.permute(0, 4, 1, 2, 3)        # → [B, 32, D, H, W]
print(x.shape)



torch.Size([128, 64, 8, 8, 8])
torch.Size([128, 8, 8, 8, 64])
torch.Size([128, 8, 8, 8, 32])
torch.Size([128, 32, 8, 8, 8])


In [ ]:
# £d input
linear_layerd = nn.Linear(in_features=64, out_features=32)
# 1st dimension (128) = batch dimension, input 64 x 64
input_2d = torch.randn(128, 64, 64)
output_2d = linear_layer_2d(input_2d)
print(output_2d.size())
# torch.Size([128, 64, 32])

In [90]:
import torch
import torch.nn as nn


class Conv2DBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_channels, D):
        super().__init__()
        self.layer1 = nn.BatchNorm2d(input_channels)
        
        self.layer1_1 = nn.Conv2d(input_channels, D, kernel_size=3, padding=1)
        self.layer1_2 = nn.ReLU() # output shape [8, 16, 27, 33, 60]
        self.layer1_3 = nn.MaxPool2d(2, padding=1)
        self.layer1_4 = nn.BatchNorm2d(D)

        self.layer2_1 = nn.Conv2d(D, 2 * D, kernel_size=3, padding=1)
        self.layer2_2 = nn.ReLU() # output shape [8, 16, 27, 33, 60]
        self.layer2_3 = nn.MaxPool2d(2, padding=1)
        self.layer2_4 = nn.BatchNorm2d(2 * D)


    def forward(self, x):
        print(x.shape)
        x = self.layer1(x)
        print(x.shape)
        x = self.layer1_1(x)
        print(x.shape)
        x = self.layer1_2(x)
        print(x.shape)
        x = self.layer1_3(x)
        print(x.shape)
        x = self.layer1_4(x)
        print(x.shape)

        x = self.layer2_1(x)
        print(x.shape)
        x = self.layer2_2(x)
        print(x.shape)
        x = self.layer2_3(x)
        print(x.shape)
        x = self.layer2_4(x)
        print(x.shape)

       
        t_comp = x.shape[2]
        x = nn.AdaptiveMaxPool2d((t_comp, 1))(x).squeeze(-1)
        print("After MaxPool2d and squeeze:", x.shape)

        return x

# Example usage
batch_size = 8
input_channels = 1  # Number of input channels
height_length = 27
# width_length = 33
T = 100

D = 16
model = Conv2DBranch(input_channels, D)
x = torch.randn(batch_size, input_channels, T, height_length)
output = model(x)


from torchinfo import summary
summary(model, input_size=((batch_size, input_channels, T, height_length))) # ie batch 8 of 2 profiles of shape (18,33)

torch.Size([8, 1, 100, 27])
torch.Size([8, 1, 100, 27])
torch.Size([8, 16, 100, 27])
torch.Size([8, 16, 100, 27])
torch.Size([8, 16, 51, 14])
torch.Size([8, 16, 51, 14])
torch.Size([8, 32, 51, 14])
torch.Size([8, 32, 51, 14])
torch.Size([8, 32, 26, 8])
torch.Size([8, 32, 26, 8])
After MaxPool2d and squeeze: torch.Size([8, 32, 26])
torch.Size([8, 1, 100, 27])
torch.Size([8, 1, 100, 27])
torch.Size([8, 16, 100, 27])
torch.Size([8, 16, 100, 27])
torch.Size([8, 16, 51, 14])
torch.Size([8, 16, 51, 14])
torch.Size([8, 32, 51, 14])
torch.Size([8, 32, 51, 14])
torch.Size([8, 32, 26, 8])
torch.Size([8, 32, 26, 8])
After MaxPool2d and squeeze: torch.Size([8, 32, 26])


Layer (type:depth-idx)                   Output Shape              Param #
Conv2DBranch                             [8, 32, 26]               --
├─BatchNorm2d: 1-1                       [8, 1, 100, 27]           2
├─Conv2d: 1-2                            [8, 16, 100, 27]          160
├─ReLU: 1-3                              [8, 16, 100, 27]          --
├─MaxPool2d: 1-4                         [8, 16, 51, 14]           --
├─BatchNorm2d: 1-5                       [8, 16, 51, 14]           32
├─Conv2d: 1-6                            [8, 32, 51, 14]           4,640
├─ReLU: 1-7                              [8, 32, 51, 14]           --
├─MaxPool2d: 1-8                         [8, 32, 26, 8]            --
├─BatchNorm2d: 1-9                       [8, 32, 26, 8]            64
Total params: 4,898
Trainable params: 4,898
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 29.96
Input size (MB): 0.09
Forward/backward pass size (MB): 5.56
Params size (MB): 0.02
Estimated Total Size (MB): 5.6

In [162]:
import torch
import torch.nn as nn


class Conv1DBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_channels, D):
        super().__init__()
        self.layer1 = nn.BatchNorm1d(input_channels)
        
        self.layer1_1 = nn.Conv1d(input_channels, D, kernel_size=3, padding=1)
        self.layer1_2 = nn.ReLU() # output shape [8, 16, 27, 33, 60]
        self.layer1_3 = nn.MaxPool1d(2, padding=1)
        self.layer1_4 = nn.BatchNorm1d(D)

        self.layer2_1 = nn.Conv1d(D, 2 * D, kernel_size=3, padding=1)
        self.layer2_2 = nn.ReLU() # output shape [8, 16, 27, 33, 60]
        self.layer2_3 = nn.MaxPool1d(2, padding=1)
        self.layer2_4 = nn.BatchNorm1d(2 * D)


    def forward(self, x):
        print('\nConv1D')
        print(x.shape)
        x = self.layer1(x)
        x = self.layer1_1(x)
        x = self.layer1_2(x)
        x = self.layer1_3(x)
        x = self.layer1_4(x)

        x = self.layer2_1(x)
        x = self.layer2_2(x)
        x = self.layer2_3(x)
        x = self.layer2_4(x)
        print(x.shape)

        return x

# Example usage
batch_size = 8
input_channels = 1  # Number of input channels
height_length = 27
# width_length = 33
T = 100

D = 16
model = Conv1DBranch(input_channels, D)
x = torch.randn(batch_size, input_channels, T)
output = model(x)


from torchinfo import summary
summary(model, input_size=((batch_size, input_channels, T))) # ie batch 8 of 2 profiles of shape (18,33)


Conv1D
torch.Size([8, 1, 100])
torch.Size([8, 32, 26])

Conv1D
torch.Size([8, 1, 100])
torch.Size([8, 32, 26])


Layer (type:depth-idx)                   Output Shape              Param #
Conv1DBranch                             [8, 32, 26]               --
├─BatchNorm1d: 1-1                       [8, 1, 100]               2
├─Conv1d: 1-2                            [8, 16, 100]              64
├─ReLU: 1-3                              [8, 16, 100]              --
├─MaxPool1d: 1-4                         [8, 16, 51]               --
├─BatchNorm1d: 1-5                       [8, 16, 51]               32
├─Conv1d: 1-6                            [8, 32, 51]               1,568
├─ReLU: 1-7                              [8, 32, 51]               --
├─MaxPool1d: 1-8                         [8, 32, 26]               --
├─BatchNorm1d: 1-9                       [8, 32, 26]               64
Total params: 1,730
Trainable params: 1,730
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.69
Input size (MB): 0.00
Forward/backward pass size (MB): 0.32
Params size (MB): 0.01
Estimated Total Size (MB): 0.33

In [163]:
import torch
import torch.nn as nn


class Conv2DBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_channels, D):
        super().__init__()
        self.layer1 = nn.BatchNorm2d(input_channels)
        
        self.layer1_1 = nn.Conv2d(input_channels, D, kernel_size=3, padding=1)
        self.layer1_2 = nn.ReLU() # output shape [8, 16, 27, 33, 60]
        self.layer1_3 = nn.MaxPool2d(2, padding=1)
        self.layer1_4 = nn.BatchNorm2d(D)

        self.layer2_1 = nn.Conv2d(D, 2 * D, kernel_size=3, padding=1)
        self.layer2_2 = nn.ReLU() # output shape [8, 16, 27, 33, 60]
        self.layer2_3 = nn.MaxPool2d(2, padding=1)
        self.layer2_4 = nn.BatchNorm2d(2 * D)


    def forward(self, x):
        print('\nConv2D')
        print(x.shape)
        x = self.layer1(x)
        x = self.layer1_1(x)
        x = self.layer1_2(x)
        x = self.layer1_3(x)
        x = self.layer1_4(x)

        x = self.layer2_1(x)

        x = self.layer2_2(x)
        x = self.layer2_3(x)
        x = self.layer2_4(x)

        t_comp = x.shape[2]
        x = nn.AdaptiveMaxPool2d((t_comp, 1))(x).squeeze(-1)
        print(x.shape)
        
        return x

# Example usage
batch_size = 8
input_channels = 1  # Number of input channels
height_length = 27
# width_length = 33
T = 100

D = 16
model = Conv2DBranch(input_channels, D)
x = torch.randn(batch_size, input_channels, T, height_length)
output = model(x)


from torchinfo import summary
summary(model, input_size=((batch_size, input_channels, T, height_length))) # ie batch 8 of 2 profiles of shape (18,33)


Conv2D
torch.Size([8, 1, 100, 27])
torch.Size([8, 32, 26])

Conv2D
torch.Size([8, 1, 100, 27])
torch.Size([8, 32, 26])


Layer (type:depth-idx)                   Output Shape              Param #
Conv2DBranch                             [8, 32, 26]               --
├─BatchNorm2d: 1-1                       [8, 1, 100, 27]           2
├─Conv2d: 1-2                            [8, 16, 100, 27]          160
├─ReLU: 1-3                              [8, 16, 100, 27]          --
├─MaxPool2d: 1-4                         [8, 16, 51, 14]           --
├─BatchNorm2d: 1-5                       [8, 16, 51, 14]           32
├─Conv2d: 1-6                            [8, 32, 51, 14]           4,640
├─ReLU: 1-7                              [8, 32, 51, 14]           --
├─MaxPool2d: 1-8                         [8, 32, 26, 8]            --
├─BatchNorm2d: 1-9                       [8, 32, 26, 8]            64
Total params: 4,898
Trainable params: 4,898
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 29.96
Input size (MB): 0.09
Forward/backward pass size (MB): 5.56
Params size (MB): 0.02
Estimated Total Size (MB): 5.6

In [164]:
import torch
import torch.nn as nn


class Conv3DBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_channels, D):
        super().__init__()
        self.layer1 = nn.BatchNorm3d(input_channels)
        
        self.layer1_1 = nn.Conv3d(input_channels, D, kernel_size=3, padding=1)
        self.layer1_2 = nn.ReLU() # output shape [8, 16, 27, 33, 60]
        self.layer1_3 = nn.MaxPool3d(2, padding=1)
        self.layer1_4 = nn.BatchNorm3d(D)

        self.layer2_1 = nn.Conv3d(D, 2 * D, kernel_size=3, padding=1)
        self.layer2_2 = nn.ReLU() # output shape [8, 16, 27, 33, 60]
        self.layer2_3 = nn.MaxPool3d(2, padding=1)
        self.layer2_4 = nn.BatchNorm3d(2 * D)


    def forward(self, x):
        print('\nConv3D')
        print(x.shape)
        x = self.layer1(x)
        x = self.layer1_1(x)
        x = self.layer1_2(x)
        x = self.layer1_3(x)
        x = self.layer1_4(x)

        x = self.layer2_1(x)
        x = self.layer2_2(x)
        x = self.layer2_3(x)
        x = self.layer2_4(x)
       
        t_comp = x.shape[2]
        x = nn.AdaptiveMaxPool3d((t_comp, 1, 1))(x).squeeze(-1).squeeze(-1)
        print(x.shape)
        
        return x

# Example usage
batch_size = 8
input_channels = 1  # Number of input channels
height_length = 27
width_length = 33
T = 100

D = 16
model = Conv3DBranch(input_channels, D)
x = torch.randn(batch_size, input_channels, T, height_length, width_length)
output = model(x)


from torchinfo import summary
summary(model, input_size=((batch_size, input_channels, T, height_length, width_length))) # ie batch 8 of 2 profiles of shape (18,33)


Conv3D
torch.Size([8, 1, 100, 27, 33])
torch.Size([8, 32, 26])

Conv3D
torch.Size([8, 1, 100, 27, 33])
torch.Size([8, 32, 26])


Layer (type:depth-idx)                   Output Shape              Param #
Conv3DBranch                             [8, 32, 26]               --
├─BatchNorm3d: 1-1                       [8, 1, 100, 27, 33]       2
├─Conv3d: 1-2                            [8, 16, 100, 27, 33]      448
├─ReLU: 1-3                              [8, 16, 100, 27, 33]      --
├─MaxPool3d: 1-4                         [8, 16, 51, 14, 17]       --
├─BatchNorm3d: 1-5                       [8, 16, 51, 14, 17]       32
├─Conv3d: 1-6                            [8, 32, 51, 14, 17]       13,856
├─ReLU: 1-7                              [8, 32, 51, 14, 17]       --
├─MaxPool3d: 1-8                         [8, 32, 26, 8, 9]         --
├─BatchNorm3d: 1-9                       [8, 32, 26, 8, 9]         64
Total params: 14,402
Trainable params: 14,402
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 1.66
Input size (MB): 2.85
Forward/backward pass size (MB): 138.06
Params size (MB): 0.06
Estimated Total Size (MB):

In [ ]:
class MultiBranchCNNModel(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_shapes, output_shape, D=16):
        super().__init__()

        self.branches = nn.ModuleList()
        self.output_shape = output_shape[0][0]
        # merged_dim = 0

        for shape in input_shapes:
            if len(shape) == 4:  # e.g., (2, T, 15, 17) images evolving in time
                input_len = shape[0]
                branch = Conv3DBranch(input_len, D)
                # merged_dim += input_len
            elif len(shape) == 3:  # e.g., (1, T, 15) profiles evolving in time
                input_len = shape[0]
                branch = Conv2DBranch(input_len, D)
                # merged_dim += input_len
            elif len(shape) == 2:  # e.g., (7, T, ) time series evolving in time
                input_len = shape[0]
                branch = Conv1DBranch(input_len, D)
                # merged_dim += shape[0]
            else:
                raise ValueError(f"Unsupported input shape: {shape}")
            self.branches.append(branch)

    # ------------------------------------------------------------------------------------------------------------------
    def forward(self, *inputs):
        branch_outputs = []

        for branch, x in zip(self.branches, inputs[0]):
            out = branch(x)
            branch_outputs.append(out)
        
        print('\nCommon Layer')     
        merged = torch.cat(branch_outputs, dim=1)
        print(merged.shape)   
        merged = Conv1DBranch(merged.shape[1], 2*D)(merged)
        merged = torch.flatten(merged,1)
        print(merged.shape)    

        self.fc = nn.Sequential(
            nn.BatchNorm1d(merged.shape[1]),
            nn.Linear(merged.shape[1], 4 * D),
            nn.ReLU(),
            nn.BatchNorm1d(4 * D),
            nn.Linear(4 * D, 2 * D),
            nn.ReLU(),
            nn.BatchNorm1d(2 * D),
            nn.Dropout(0.2),
            nn.Linear(2 * D, self.output_shape),
        )
        print('\nDNN Layer')    
        print(merged.shape)    
        merged = self.fc(merged)
        print(merged.shape)    

        return merged

    # ------------------------------------------------------------------------------------------------------------------


In [168]:
# Example usage
batch_size = 8

D = 16
T = 100

input_channels_1 = 5  # Number of input channels

input_channels_2 = 1  
height_length_2 = 54

input_channels_3 = 1  # Number of input channels
height_length_3 = 27
width_length_3 = 33

output_shape = [[7]]

x_init = [ torch.randn(input_channels_1, T), # time series evolving in time
      torch.randn(input_channels_2, T, height_length_2), # profiles evolving in time
      torch.randn(input_channels_3, T, height_length_3, width_length_3) # images evolving in time
      ]

model = MultiBranchCNNModel([arr.shape for arr in x_init], output_shape, D)

x = [ torch.randn(batch_size, input_channels_1, T), # time series evolving in time
      torch.randn(batch_size, input_channels_2, T, height_length_2), # profiles evolving in time
      torch.randn(batch_size, input_channels_3, T, height_length_3, width_length_3) # images evolving in time
]

output = model(x)



Conv1D
torch.Size([8, 5, 100])
torch.Size([8, 32, 26])

Conv2D
torch.Size([8, 1, 100, 54])
torch.Size([8, 32, 26])

Conv3D
torch.Size([8, 1, 100, 27, 33])
torch.Size([8, 32, 26])

Common Layer
torch.Size([8, 96, 26])

Conv1D
torch.Size([8, 96, 26])
torch.Size([8, 64, 8])
torch.Size([8, 512])

DNN Layer
torch.Size([8, 512])
torch.Size([8, 7])


In [146]:
output.shape

torch.Size([8, 7])

In [170]:
[arr.shape for arr in x_init]

[torch.Size([5, 100]), torch.Size([1, 100, 54]), torch.Size([1, 100, 27, 33])]